### Extract Drama, Film, Cartoon List From Kaggle

### Drama List

In [ ]:
import pandas as pd
import re

def extract_year(series):
    """
    다양한 형식(날짜, 소수점 포함 숫자 등)에서 4자리 연도만 추출하여 
    '2025'와 같은 정수형 문자열 포맷으로 반환하는 함수
    """
    # 1. 데이터를 문자열로 변환하고 .0과 같은 소수점 제거
    series = series.astype(str).str.replace(r'\.0$', '', regex=True)
    
    # 2. pandas datetime 변환 시도 (표준 날짜 형식 처리)
    dt_series = pd.to_datetime(series, errors='coerce', dayfirst=False)
    years = dt_series.dt.year
    
    # 3. datetime 변환 실패한 경우나 원본에서 정규식으로 4자리 숫자 추출
    # (예: "2025-\N", "15/10/2022", "2021-2025")
    extracted = series.str.extract(r'(\d{4})')[0]
    
    # 4. datetime 결과가 있으면 그것을 쓰고, 없으면 정규식 추출 결과 사용
    final_years = years.fillna(pd.to_numeric(extracted, errors='coerce'))
    
    # 5. 최종 결과를 '2022' 형태의 문자열로 변환 (결측치는 공백)
    return final_years.apply(lambda x: str(int(x)) if pd.notnull(x) else '')

def process_kdrama_files(folder):
    all_data = {}

    # 1. kdrama.csv (이미 연도인 경우 처리)
    df1 = pd.read_csv(folder + 'kdrama.csv')
    df1 = df1.rename(columns={"Year of release": "Year"})
    df1['Year'] = extract_year(df1['Year'])
    all_data['kdrama.csv'] = df1

    # 2. kdrama_list.csv
    df2 = pd.read_csv(folder + 'kdrama_list.csv')
    all_data['kdrama_list.csv'] = df2
    df2['Year'] = df2['Year'].astype(str)

    # 3. kdramalist.csv (10-Sep-18-30-Oct-18 형식 처리)
    df_kl = pd.read_csv(folder + 'kdramalist.csv')
    df_kl = df_kl.rename(columns={"drama_name": "Name"})
    df_kl['Year'] = extract_year(df_kl['start airing'])
    all_data['kdramalist.csv'] = df_kl[['Name', 'Year']]

    # 4. kdramas.csv (2025-\N, 2021-2025 형식 처리)
    df_kd = pd.read_csv(folder + 'kdramas.csv')
    df_kd = df_kd.rename(columns={"title": "Name"})
    df_kd['Year'] = extract_year(df_kd['startYear'])
    all_data['kdramas.csv'] = df_kd[['Name', 'Year']]

    # 5. korean_drama.csv
    df5 = pd.read_csv(folder + 'korean_drama.csv')
    df5 = df5.rename(columns={"drama_name": "Name", "year": "Year"})
    df5['Year'] = extract_year(df5['Year'])
    all_data['korean_drama.csv'] = df5

    # 6. TMDB_tv_dataset_v3.csv (2011-04-17-2019-05-19 형식 처리)
    df_tmdb = pd.read_csv(folder + 'TMDB_tv_dataset_v3.csv')
    df_tmdb = df_tmdb[df_tmdb['vote_count'] >= 1000].copy()
    df_tmdb = df_tmdb.rename(columns={"name": "Name"})
    df_tmdb['Year'] = extract_year(df_tmdb['first_air_date'])
    all_data['TMDB_tv_dataset_v3.csv'] = df_tmdb[['Name', 'Year']]

    # 7. Top_15_Korean_Dramas_2020_2025.csv (2025-07-03, 15/10/2022 형식 처리)
    df_top15 = pd.read_csv(folder + 'Top_15_Korean_Dramas_2020_2025.csv')
    df_top15 = df_top15.rename(columns={"Title": "Name", "Release Date": "Year"})
    df_top15['Year'] = extract_year(df_top15['Year'])
    all_data['Top_15_Korean_Dramas_2020_2025.csv'] = df_top15[['Name', 'Year']]

    # 8. top_5000_dramas.csv
    df8 = pd.read_csv(folder + 'top_5000_dramas.csv')
    df8 = df8.rename(columns={"Released Year": "Year"})
    df8['Year'] = extract_year(df8['Year'])
    all_data['top_5000_dramas.csv'] = df8

    # 9. top_100_kdrama.csv
    df9 = pd.read_csv(folder + 'top100_kdrama.csv')
    df9 = df9.rename(columns={"Year of release": "Year"})
    df9['Year'] = extract_year(df9['Year'])
    all_data['top_100_kdrama.csv'] = df9

    # 10. toptvshow_Cleaned_v2.csv
    df10 = pd.read_csv(folder + 'toptvshow_Cleaned_v2.csv')
    df10 = df10.rename(columns={"tv_show": "Name", "first_season_yr": "Year"})
    df10['Year'] = extract_year(df10['Year'])
    all_data['toptvshow_Cleaned_v2.csv'] = df10

    # 데이터 통합
    combined_df = pd.concat(all_data, ignore_index=True)

    final_df = combined_df[combined_df['Year'] != ''].copy()
    final_df = final_df.dropna(subset=['Year'])

    final_df = final_df.drop_duplicates(subset=['Name'], keep='first').reset_index(drop=True)
    final_df = final_df.sort_values(by=['Year', 'Name'], ascending=[True, True]).reset_index(drop=True)
    final_df = final_df[['Name', 'Year']]

    # 3. 최종 리스트 생성
    final_name_list = final_df['Name'].tolist()
    final_year_list = final_df['Year'].tolist()

    print(f"--- 통합 결과 ---")
    print(f"Total Unique Names: {len(final_df)}")
    print(f"Names (First 5): {final_name_list[:5]}")
    print(f"Years (First 5): {final_year_list[:5]}")

    return final_df, final_name_list, final_year_list

folder = "./드라마/"
# 폴더 경로 끝에 '/'가 있는지 확인 필요
drama_df, names, years = process_kdrama_files(folder)
drama_df.to_csv(folder + "total_drama.csv")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17740\2372147681.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_series = pd.to_datetime(series, errors='coerce', dayfirst=False)


--- 통합 결과 ---
Total Unique Names: 8286
Names (First 5): ['The Flintstones', 'Star Trek', 'Frog husband', 'Scooby Doo, Where Are You!', 'My lady']
Years (First 5): ['1960', '1966', '1969', '1969', '1970']


### Film List

In [41]:
import pandas as pd
import re

def extract_year(series):
    """
    다양한 형식(날짜, 소수점 포함 숫자 등)에서 4자리 연도만 추출하여 
    '2025'와 같은 정수형 문자열 포맷으로 반환하는 함수
    """
    # 1. 데이터를 문자열로 변환하고 .0과 같은 소수점 제거
    series = series.astype(str).str.replace(r'\.0$', '', regex=True)
    
    # 2. pandas datetime 변환 시도 (표준 날짜 형식 처리)
    dt_series = pd.to_datetime(series, errors='coerce', dayfirst=False)
    years = dt_series.dt.year
    
    # 3. datetime 변환 실패한 경우나 원본에서 정규식으로 4자리 숫자 추출
    # (예: "2025-\N", "15/10/2022", "2021-2025")
    extracted = series.str.extract(r'(\d{4})')[0]
    
    # 4. datetime 결과가 있으면 그것을 쓰고, 없으면 정규식 추출 결과 사용
    final_years = years.fillna(pd.to_numeric(extracted, errors='coerce'))
    
    # 5. 최종 결과를 '2022' 형태의 문자열로 변환 (결측치는 공백)
    return final_years.apply(lambda x: str(int(x)) if pd.notnull(x) else '')

def process_film_files(folder):
    all_data = {}

    # 1. Animation_Movies.csv 
    df1 = pd.read_csv(folder + 'Animation_Movies.csv')
    df1 = df1[df1['vote_count'] >= 1000].copy()
    df1 = df1.rename(columns={"title": "Name", "release_date": "Year"})
    df1['Year'] = extract_year(df1['Year'])
    all_data['Animation_Movies.csv'] = df1

    # 2. IMDB-Movie-Data.csv
    df2 = pd.read_csv(folder + 'IMDB-Movie-Data.csv')
    df2 = df2.rename(columns={"Title": "Name"})
    df2['Year'] = extract_year(df2['Year'])
    all_data['IMDB-Movie-Data.csv'] = df2

    # 3. movie_history_list.csv 
    df3 = pd.read_csv(folder + 'movie_history_list.csv')
    df3 = df3.rename(columns={"Title": "Name"})
    df3['Year'] = extract_year(df3['Year'])
    all_data['movie_history_list.csv'] = df3

    # 4. top_movies.csv 
    df4 = pd.read_csv(folder + 'top_movies.csv')
    df4 = df4.rename(columns={"Title": "Name"})
    df4['Year'] = extract_year(df4['Year'])
    all_data['top_movies.csv'] = df4

    # 5. top_rated_2000webseries.csv 
    df5 = pd.read_csv(folder + 'top_rated_2000webseries.csv')
    df5 = df5.rename(columns={"title": "Name", "premiere_date": "Year"})
    df5['Year'] = extract_year(df5['Year'])
    all_data['top_rated_2000webseries.csv'] = df5

    # 6. top_rated_9000_movies_on_TMDB.csv 
    df6 = pd.read_csv(folder + 'top_rated_9000_movies_on_TMDB.csv')
    df6 = df6.rename(columns={"title": "Name", "release_date": "Year"})
    df6['Year'] = extract_year(df6['Year'])
    all_data['top_rated_9000_movies_on_TMDB.csv'] = df6

    # 데이터 통합
    combined_df = pd.concat(all_data, ignore_index=True)

    final_df = combined_df[combined_df['Year'] != ''].copy()
    final_df = final_df.dropna(subset=['Year'])

    # pattern = 'д|ä|Š'
    # final_df = final_df[~final_df['Name'].str.contains(pattern, na=False, regex=True)]

    final_df = final_df.drop_duplicates(subset=['Name'], keep='first').reset_index(drop=True)
    final_df = final_df.sort_values(by=['Year', 'Name'], ascending=[True, True]).reset_index(drop=True)
    final_df = final_df[['Name', 'Year']]

    # 3. 최종 리스트 생성
    final_name_list = final_df['Name'].tolist()
    final_year_list = final_df['Year'].tolist()

    print(f"--- 통합 결과 ---")
    print(f"Total Unique Names: {len(final_df)}")
    print(f"Names (First 5): {final_name_list[:5]}")
    print(f"Years (First 5): {final_year_list[:5]}")

    return final_df, final_name_list, final_year_list

folder = "./영화/"
# 폴더 경로 끝에 '/'가 있는지 확인 필요
film_df, names, years = process_film_files(folder)
film_df.to_csv(folder + "total_film.csv")

--- 통합 결과 ---
Total Unique Names: 11761
Names (First 5): ['A Trip to the Moon', 'A Trip to the Moon (Le Voyage Dans La Lune)', 'The Great Train Robbery', 'Les Vampires', 'The Birth of a Nation']
Years (First 5): ['1902', '1902', '1903', '1915', '1915']


### Cartoon List

In [42]:
import pandas as pd
import re

def process_cartoon_files(folder):
    all_data = {}

    # 1. comic_books_10000_dataset.csv 
    df1 = pd.read_csv(folder + 'comic_books_10000_dataset.csv')
    df1 = df1.rename(columns={"Title": "Name", "Release Year": "Year"})
    all_data['comic_books_10000_dataset.csv'] = df1

    # 데이터 통합
    combined_df = pd.concat(all_data, ignore_index=True)

    final_df = combined_df[combined_df['Year'] != ''].copy()
    final_df = final_df.dropna(subset=['Year'])

    final_df = final_df.sort_values(by=['Year', 'Name'], ascending=[True, True]).reset_index(drop=True)
    final_df = final_df[['Name', 'Year']]

    # 3. 최종 리스트 생성
    final_name_list = final_df['Name'].tolist()
    final_year_list = final_df['Year'].tolist()

    print(f"--- 통합 결과 ---")
    print(f"Total Unique Names: {len(final_df)}")
    print(f"Names (First 5): {final_name_list[:5]}")
    print(f"Years (First 5): {final_year_list[:5]}")

    return final_df, final_name_list, final_year_list

folder = "./영화/"
# 폴더 경로 끝에 '/'가 있는지 확인 필요
cartoon_df, names, years = process_cartoon_files(folder)
cartoon_df.to_csv(folder + "total_cartoon.csv")

--- 통합 결과 ---
Total Unique Names: 10000
Names (First 5): ['100 Bullets', '100-Page Giant Justice Society', '100-Page Giant The Flash', 'Abyss Infinity', 'Abyss Legend']
Years (First 5): [2000, 2000, 2000, 2000, 2000]
